## 欢迎来到第六周

史诗般的最终周

以及

# 欢迎来到 **M**ODEL **C**ONTEXT **P**ROTOCOL（模型上下文协议）！

欢迎回来继续学习 OpenAI Agents SDK ❤️❤️❤️

### 请注意

由于我一直在更新内容，这里可能与视频有所不同！

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">致 Windows PC 用户 — 重要公告</h2>
            <span style="color:#ff7800;">我有一个不太好的消息。在 Windows PC 上运行 MCP Server 存在问题；Mac 和 Linux 则没有问题。这是截至 2025 年 5 月 4 日的已知问题。我让 o3 配合 Deep Research 尝试寻找解决方案；它<a href="https://chatgpt.com/share/6817bbc3-3d0c-8012-9b51-631842470628">确认了这个问题</a>并验证了可行的解决方案。<br/><br/>
            解决方案有点麻烦。需要利用"WSL"，即微软提供的在 PC 上运行 Linux 的方式。你需要执行更多的安装步骤！但这很快就能完成，多位学生已经确认这个方案对他们来说完美有效，之后第六周的 MCP 实验就能正常运行了。而且，WSL 实际上是在 Windows PC 上开发软件的绝佳方式。<br/>
            WSL 的安装说明在 Setup 文件夹中，<a href="../setup/SETUP-WSL.md">请查看名为 SETUP-WSL.md 的文件</a>。我希望这只是一点小耽搁 — 你应该很快就能重新上手。这就是使用前沿技术的乐趣！<br/><br/>
            非常感谢 Markus、Abhi、Hui-Ling 等多位同学，帮助我排查并确认了这个修复方案。
            </span>
        </td>
    </tr>
</table>

In [ ]:
# 导入库

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os

In [ ]:
load_dotenv(override=True)

### 在 OpenAI Agents SDK 中使用 MCP

1. 创建一个客户端

2. 让它启动一个服务器

3. 收集服务器可以使用的工具

让我们来试试上周看过的 Fetch MCP 服务器

### 特别说明 — 如果你更新了 OpenAI Agents 框架：

OpenAI Agents SDK 最近升级了 SDK 并做了一个破坏性变更（谢谢你，OpenAI 😂）。我还没有将这个仓库中的版本更新到最新版本。如果你自己升级了这个包，可能会遇到缺少位置参数的错误。如果遇到这个错误，你需要将 `await server.list_tools()` 替换为 `await server.session.list_tools()`，并将最后一行的 `fetch_tools` 改为 `fetch_tools.tools`。而且每次遇到这个错误都需要做同样的修改！这就是走在技术前沿的乐趣……

In [ ]:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    fetch_tools = await server.list_tools()

fetch_tools

## 额外安装步骤 — 如果你的电脑上没有 Node 和 Playwright

下一个 MCP 工具使用 node（JavaScript 服务器），需要你在电脑上安装 `npx` 命令。

额外的安装步骤请参见：

[Node 和 Playwright 安装说明](../setup/SETUP-node.md)

## 现在再来 2 个 MCP 服务器 — 这次是基于 JavaScript 的，使用 node

### 如果下一个代码单元卡住了，请参见上面链接的 SETUP-node 说明末尾的重要故障排除部分。

In [ ]:

playwright_params = {"command": "npx","args": [ "@playwright/mcp@latest"]}

async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as server:
    playwright_tools = await server.list_tools()

playwright_tools


In [ ]:

sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

async with MCPServerStdio(params=files_params,client_session_timeout_seconds=60) as server:
    file_tools = await server.list_tools()

file_tools

### 现在……让 Agent 配合工具登场！

In [ ]:
instructions = """
你通过浏览互联网来完成你的指令。
你非常擅长独立浏览互联网来完成任务，
包括接受所有 Cookie 以及适当地点击"暂不"来获取你需要的内容。
如果一个网站没有收获，就试试另一个。
坚持不懈直到完成你的任务，
根据需要尝试不同的选项和网站。
当你需要写入文件时，只能在 sandbox 文件夹内操作。
"""


async with MCPServerStdio(params=files_params, client_session_timeout_seconds=60) as mcp_server_files:
    async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as mcp_server_browser:
        agent = Agent(
            name="investigator", 
            instructions=instructions, 
            model="gpt-4.1-mini",
            mcp_servers=[mcp_server_files, mcp_server_browser]
            )
        with trace("investigate"):
            result = await Runner.run(agent, "找到一个很棒的香蕉太妃派食谱，然后用 markdown 格式总结到 banoffee.md 文件中")
            print(result.final_output)

### 查看 Trace 追踪

https://platform.openai.com/traces

### 现在来看看一些 MCP 市场

https://mcp.so

https://glama.ai/mcp

https://smithery.ai/

https://huggingface.co/blog/LLMhacker/top-11-essential-mcp-libraries

HuggingFace 优质社区文章：
https://huggingface.co/blog/Kseniase/mcp